# Template Notebook

This notebook includes a code cell with the following functionality:

 - **Version Check Cell**: This cell checks if the notebook is up-to-date with the latest version available on provided GitHub repository.

## Version Check Cell

In [ ]:
# This code allows to have a version tracking
current_version = "0.0.1"
notebook_name = "Notebook_template"  # Replace with the actual notebook name

# First of all check if ipywidgets is installed, if not through an informative error
try:
    import ipywidgets as widgets
except ImportError:
    raise ImportError("ipywidgets is not installed. Please install it using 'pip install ipywidgets' or 'conda install -c conda-forge ipywidgets'.")

try:
    import yaml
except ImportError:
    raise ImportError("pyyaml is not installed. Please install it using 'pip install pyyaml' or 'conda install -c conda-forge pyyaml'.")

import requests
import base64

# Create widgets
github_repo = widgets.Text(
    value="",
    description="*GitHub Repository URL:",
    placeholder="e.g https://github.com/username/repository",
    layout=widgets.Layout(width="70%"),
    style={'description_width': '150px'},
)

github_token = widgets.Password(
    value="",
    description="*Personal Access Token:",
    placeholder="[Disabled] e.g. ghp_XXXXXXXXXXXXXXXXXXXX",
    disabled=True,
    layout=widgets.Layout(width="70%"),
    style={'description_width': '150px'},
)

test_button = widgets.Button(
    description="Test Connection",
    button_style='success',
)

output_area = widgets.HTML()

# By default the repository is considered public
if "repository_is_private" not in globals():
    repository_is_private = False
elif repository_is_private:
    github_token.disabled = False
    github_token.placeholder = "e.g. ghp_XXXXXXXXXXXXXXXXXXXX"

def on_test_button_clicked(b):
    global repository_is_private
    
    # Check that the GitHub repository URL is provided
    if not github_repo.value:
        output_area.value = "⚠️ Please provide a GitHub repository URL."
        return
    
    # Get the owner and repo name from the URL
    repo_url = github_repo.value.rstrip('/')
    github_owner, github_repo_name = repo_url.split('/')[-2:]
    
    # Online version checking file path
    version_file_path = "notebooks/notebook_latest_versions.yaml"
    version_url = f"https://api.github.com/repos/{github_owner}/{github_repo_name}/contents/{version_file_path}"

    # Do an initial request to check if the repository is public
    if not repository_is_private:
        version_response = requests.get(version_url)
        if version_response.status_code == 404:
            repository_is_private = True
            github_token.disabled = False
            github_token.placeholder = "e.g. ghp_XXXXXXXXXXXXXXXXXXXX"
            output_area.value = f"⚠️ We have detected that the repository might be private. Please provide a Personal Access Token and click 'Test Connection' again."
            return
    else:
        headers = {"Accept": "application/vnd.github.v3+json"}
        if not github_token.value:
            output_area.value = "⚠️ Personal Access Token is required for private repositories."
            return
        headers["Authorization"] = f"token {github_token.value}"

        version_response = requests.get(version_url, headers=headers)

    # Check the response status
    if version_response.status_code == 200:
        content = version_response.json()['content']
        decoded_content = base64.b64decode(content).decode('utf-8')
        config = yaml.safe_load(decoded_content)
        latest_version = config.get(notebook_name, "")

        output_area.value = (f"<b>Notebook version:</b> `{current_version}`<br>"
                             f"<b>Latest version available:</b> `{latest_version}`<br>")

        if latest_version == "":
            output_area.value += "⚠️ This notebook is not listed in the version file.<br>"
        elif current_version == latest_version:
            output_area.value += "✅ This notebook is up-to-date.<br>"
        else:
            output_area.value += f"⚠️ A new version of this notebook is available."
    else:
        output_area.value += "⚠️ Could not retrieve the version file.<br>"

test_button.on_click(on_test_button_clicked)
# Widget layout
widget_box = widgets.VBox([github_repo, github_token, test_button, output_area])
display(widget_box)